# ProcessOptimizer KZ — Model Validation
Loads registered models from the MLflow registry and runs a full validation suite:
- ROC / PR curves for each classifier
- Time-series precision recall (anomaly detection @ multiple thresholds)
- Sliding-window forecaster horizon comparison
- Walk-forward cross-validation (time-series CV)
- All results logged back to MLflow experiment `plant_alarm_validation`

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import joblib
import mlflow
import mlflow.pyfunc
from mlflow.tracking import MlflowClient

from sklearn.metrics import (
    roc_curve, auc, precision_recall_curve, average_precision_score,
    f1_score, precision_score, recall_score, roc_auc_score,
    classification_report, confusion_matrix, brier_score_loss,
)
from sklearn.model_selection import TimeSeriesSplit

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')
os.makedirs('eda_plots', exist_ok=True)

MLFLOW_URI  = os.getenv('MLFLOW_TRACKING_URI', 'http://localhost:5000')
VAL_EXP     = 'plant_alarm_validation'

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(VAL_EXP)
client = MlflowClient(MLFLOW_URI)
print(f'MLflow tracking URI : {MLFLOW_URI}')
print(f'Experiment          : {VAL_EXP}')

## Load Data & Rebuild Labels

In [ ]:
df = pd.read_csv('data_drop/labeled_dataset.csv',
                 index_col='timestamp', parse_dates=True)
df = df.loc[:, ~df.columns.str.startswith('Unnamed')]

# Recompute SP-PV deviations — must match 02_models.ipynb exactly
df['LIC31012_deviation'] = df['LIC31012_SP'] - df['LIC31012_PV']
df['LIC31002_deviation'] = df['LIC31002_SP'] - df['LIC31002_PV']
df['FIC31011_deviation'] = df['FIC31011_SP'] - df['FIC31011_PV']

# POINT_FEATURES must be identical to 02_models.ipynb
POINT_FEATURES = [
    'TE301020', 'PDT31008', 'PDT31001', 'PDT31007',
    'FQ31050', 'LT301031',
    'LIC31012_PV', 'LIC31002_PV', 'FIC31011_PV',
    'LIC31012_OP', 'LIC31002_OP', 'FIC31011_OP',
    'LIC31012_deviation', 'LIC31002_deviation', 'FIC31011_deviation',
    'hour_of_day', 'day_of_week',
]
SENSOR_COLS = ['TE301020', 'PDT31008', 'PDT31001', 'PDT31007',
               'FQ31050', 'LT301031', 'LIC31012_PV', 'LIC31002_PV', 'FIC31011_PV']

from sklearn.preprocessing import StandardScaler
scaler    = joblib.load('models/scaler.joblib')
X_all     = scaler.transform(df[POINT_FEATURES].values)
y_all     = df['anomaly_label'].values

split_idx  = int(len(df) * 0.75)
X_test     = X_all[split_idx:]
y_test     = y_all[split_idx:]
test_index = df.index[split_idx:]

print(f'Full dataset : {df.shape}')
print(f'Test window  : {len(X_test)} rows  ({test_index[0]} → {test_index[-1]})')
print(f'Anomaly rate (test): {y_test.mean()*100:.1f}%')

## Load Models from Local Files
*(Falls back to local joblib if MLflow is not reachable — useful in offline dev)*

In [ ]:
def load_model_safe(registry_name, fallback_path):
    """Try MLflow registry first, fall back to local joblib."""
    for uri in [f'models:/{registry_name}@champion',
                f'models:/{registry_name}/latest']:
        try:
            m = mlflow.pyfunc.load_model(uri)
            print(f'  Loaded from registry: {uri}')
            return m, 'mlflow'
        except Exception:
            pass
    if os.path.exists(fallback_path):
        m = joblib.load(fallback_path)
        print(f'  Loaded from local file: {fallback_path}')
        return m, 'local'
    raise FileNotFoundError(f'Could not load {registry_name} from registry or {fallback_path}')


print('Loading models...')
iso_model,  iso_src  = load_model_safe('plant_alarm_isolation_forest',
                                        'models/isolation_forest.joblib')
clf_model,  clf_src  = load_model_safe('plant_alarm_regime_classifier',
                                        'models/regime_classifier.joblib')
fc1_model,  fc1_src  = load_model_safe('plant_alarm_forecaster_1h',
                                        'models/forecaster_1h.joblib')
fc3_model,  fc3_src  = load_model_safe('plant_alarm_forecaster_3h',
                                        'models/forecaster_3h.joblib')
fc6_model,  fc6_src  = load_model_safe('plant_alarm_forecaster_6h',
                                        'models/forecaster_6h.joblib')

# Soft alarm blend components (loaded separately — pyfunc registry wraps them differently)
lr_alarm_local  = (joblib.load('models/lr_soft_alarm.joblib')
                   if os.path.exists('models/lr_soft_alarm.joblib') else None)
rf_alarm_local  = (joblib.load('models/rf_soft_alarm.joblib')
                   if os.path.exists('models/rf_soft_alarm.joblib') else None)
if lr_alarm_local and rf_alarm_local:
    print('  Loaded soft alarm blend from local files')
else:
    print('  WARNING: soft alarm models not found — run 02_models.ipynb first')

print('All models loaded.')

## Isolation Forest — ROC & PR Curves

In [ ]:
# Score the full dataset, evaluate on held-out test slice
if iso_src == 'mlflow':
    scores_all = iso_model.predict(pd.DataFrame(X_all, columns=POINT_FEATURES))
    scores_all = np.array(scores_all, dtype=float).flatten()
else:
    raw = iso_model.score_samples(X_all)
    scores_all = 1 - (raw - raw.min()) / (raw.max() - raw.min())

scores_test = scores_all[split_idx:]

fpr, tpr, roc_thr = roc_curve(y_test, scores_test)
roc_auc_if = auc(fpr, tpr)

prec_c, rec_c, pr_thr = precision_recall_curve(y_test, scores_test)
ap_if = average_precision_score(y_test, scores_test)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Isolation Forest — ROC & Precision-Recall Curves', fontweight='bold')

axes[0].plot(fpr, tpr, color='#3498DB', linewidth=2, label=f'AUC = {roc_auc_if:.3f}')
axes[0].plot([0,1],[0,1],'--', color='gray', linewidth=1)
axes[0].set_xlabel('False Positive Rate'); axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(rec_c, prec_c, color='#E74C3C', linewidth=2, label=f'AP = {ap_if:.3f}')
axes[1].axhline(y_test.mean(), color='gray', linestyle='--', linewidth=1, label='Baseline')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('eda_plots/14_isolation_forest_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Isolation Forest  ROC-AUC={roc_auc_if:.3f}  AP={ap_if:.3f}')

## Threshold Sweep — F1 vs Threshold

In [ ]:
thresholds = np.linspace(0.1, 0.9, 50)
f1_scores  = [f1_score(y_test, (scores_test >= t).astype(int), zero_division=0)
              for t in thresholds]
best_thr   = thresholds[np.argmax(f1_scores)]
best_f1    = max(f1_scores)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(thresholds, f1_scores, color='#3498DB', linewidth=2)
ax.axvline(best_thr, color='red', linewidth=1.5, linestyle='--',
           label=f'Best threshold = {best_thr:.2f}  (F1 = {best_f1:.3f})')
ax.set_xlabel('Score threshold'); ax.set_ylabel('F1 score')
ax.set_title('Isolation Forest — F1 vs Decision Threshold', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('eda_plots/15_threshold_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Best threshold : {best_thr:.2f}  →  F1={best_f1:.3f}')
best_pred = (scores_test >= best_thr).astype(int)
print(classification_report(y_test, best_pred, target_names=['Normal','Alarm'], digits=3))

## Soft Alarm Blend — Calibration & ROC Curves
Evaluates the LR + Calibrated RF blend that replaces the old uncalibrated XGBoost.

In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

if lr_alarm_local and rf_alarm_local:
    lr_p_all   = lr_alarm_local.predict_proba(X_all)[:, 1]
    rf_p_all   = rf_alarm_local.predict_proba(X_all)[:, 1]
    blend_all  = 0.5 * lr_p_all + 0.5 * rf_p_all
    blend_smooth = pd.Series(blend_all, index=df.index).ewm(alpha=0.3).mean().values

    lr_p_test  = lr_p_all[split_idx:]
    rf_p_test  = rf_p_all[split_idx:]
    blend_test = blend_all[split_idx:]

    print('Soft Alarm Blend (test set):')
    for name, p in [('LR', lr_p_test), ('RF+Cal', rf_p_test), ('Blend', blend_test)]:
        from sklearn.metrics import roc_auc_score as roc_
        auc_v   = roc_(y_test, p)
        brier_v = brier_score_loss(y_test, p)
        print(f'  {name:<10} AUC={auc_v:.3f}  Brier={brier_v:.3f}')

    # IF scores for comparison
    raw_if    = iso_model.score_samples(X_all)
    if_scores = 1.0 - (raw_if - raw_if.min()) / (raw_if.max() - raw_if.min())

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Soft Alarm — Calibration vs Isolation Forest', fontweight='bold')

    ax = axes[0]
    for name, probs in [('Isolation Forest', if_scores),
                        ('LR', lr_p_all), ('RF+Cal', rf_p_all),
                        ('Blend+EWM', blend_smooth)]:
        frac, mean_p = calibration_curve(y_all, probs, n_bins=8)
        ax.plot(mean_p, frac, 's-', label=name, markersize=5)
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Perfect')
    ax.set_xlabel('Mean predicted probability'); ax.set_ylabel('Fraction positives')
    ax.set_title('Calibration Curves'); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

    ax = axes[1]
    fpr_b, tpr_b, _ = roc_curve(y_test, blend_test)
    fpr_i, tpr_i, _ = roc_curve(y_test, if_scores[split_idx:])
    ax.plot(fpr_b, tpr_b, linewidth=2,
            label=f'Blend AUC={roc_auc_score(y_test, blend_test):.3f}')
    ax.plot(fpr_i, tpr_i, linewidth=2, linestyle='--',
            label=f'IF AUC={roc_auc_score(y_test, if_scores[split_idx:]):.3f}')
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1)
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title('ROC — Blend vs IF (test set)'); ax.legend(); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('eda_plots/17_soft_alarm_calibration.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Soft alarm models not loaded — skipping this section.')

## Walk-Forward Cross-Validation (Time-Series CV)

In [ ]:
from sklearn.ensemble import IsolationForest as IF

tscv = TimeSeriesSplit(n_splits=5, gap=0)
cv_results = []

for fold, (tr_idx, te_idx) in enumerate(tscv.split(X_all)):
    X_cv_tr, X_cv_te = X_all[tr_idx], X_all[te_idx]
    y_cv_tr, y_cv_te = y_all[tr_idx], y_all[te_idx]

    cv_model = IF(
        n_estimators=200,
        contamination=float(y_cv_tr.mean()) if y_cv_tr.mean() > 0 else 0.1,
        random_state=42, n_jobs=-1
    )
    cv_model.fit(X_cv_tr)
    raw = cv_model.score_samples(X_cv_te)
    sc  = 1 - (raw - raw.min()) / (raw.max() - raw.min())
    pred = (sc >= 0.5).astype(int)

    cv_results.append({
        'fold': fold + 1,
        'train_size': len(tr_idx),
        'test_size':  len(te_idx),
        'f1':        f1_score(y_cv_te, pred, zero_division=0),
        'precision': precision_score(y_cv_te, pred, zero_division=0),
        'recall':    recall_score(y_cv_te, pred, zero_division=0),
        'roc_auc':   roc_auc_score(y_cv_te, sc) if len(np.unique(y_cv_te)) > 1 else 0.5,
    })

cv_df = pd.DataFrame(cv_results)
print('Walk-Forward CV Results:')
print(cv_df.to_string(index=False))
print(f"\nMean F1    : {cv_df['f1'].mean():.3f} ± {cv_df['f1'].std():.3f}")
print(f"Mean ROC-AUC: {cv_df['roc_auc'].mean():.3f} ± {cv_df['roc_auc'].std():.3f}")

## Forecaster Horizon Comparison

In [ ]:
def create_sliding_windows(df_in, feature_cols, label_col, window_size=12, horizon=1):
    """Matches create_sliding_windows() in 02_models.ipynb exactly.

    Returns flattened window + per-sensor mean, std, and trend (last minus first).
    Using w_trend instead of w_max avoids any ambiguity about future-value leakage
    and better captures directional sensor movement leading up to an alarm.
    """
    X_win, y_win, ts_win = [], [], []
    arr    = df_in[feature_cols].to_numpy(dtype=np.float32)
    labels = df_in[label_col].values
    index  = df_in.index
    for i in range(window_size, len(df_in) - horizon + 1):
        window  = arr[i - window_size : i]
        flat    = window.flatten()
        w_mean  = window.mean(axis=0)
        w_std   = window.std(axis=0)
        w_trend = window[-1] - window[0]   # directional change over window
        X_win.append(np.concatenate([flat, w_mean, w_std, w_trend]))
        y_win.append(int(labels[i + horizon - 1]))
        ts_win.append(index[i])
    return (np.array(X_win, dtype=np.float32),
            np.array(y_win,  dtype=np.int32),
            ts_win)


WINDOW_SIZE = 12
HORIZONS    = [1, 3, 6]
fc_local = {
    1: joblib.load('models/forecaster_1h.joblib') if os.path.exists('models/forecaster_1h.joblib') else None,
    3: joblib.load('models/forecaster_3h.joblib') if os.path.exists('models/forecaster_3h.joblib') else None,
    6: joblib.load('models/forecaster_6h.joblib') if os.path.exists('models/forecaster_6h.joblib') else None,
}

horizon_metrics = []
for h in HORIZONS:
    X_w, y_w, ts_w = create_sliding_windows(df, SENSOR_COLS, 'anomaly_label',
                                             window_size=WINDOW_SIZE, horizon=h)
    split_w = int(len(X_w) * 0.75)
    X_te_w  = X_w[split_w:]
    y_te_w  = y_w[split_w:]

    m = fc_local[h]
    if m is None:
        print(f'  Skipping h={h} — model not found')
        continue

    probs = m.predict_proba(X_te_w)[:, 1]
    preds = (probs >= 0.5).astype(int)
    horizon_metrics.append({
        'horizon_h': h,
        'f1':        f1_score(y_te_w, preds, zero_division=0),
        'precision': precision_score(y_te_w, preds, zero_division=0),
        'recall':    recall_score(y_te_w, preds, zero_division=0),
        'roc_auc':   roc_auc_score(y_te_w, probs) if len(np.unique(y_te_w)) > 1 else 0.5,
        'brier':     brier_score_loss(y_te_w, probs),
    })

hm_df = pd.DataFrame(horizon_metrics)
print('Forecaster metrics by horizon:')
print(hm_df.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Sliding-Window Forecaster — Horizon Comparison', fontweight='bold')
for metric, color, ax in [('f1', '#3498DB', axes[0]),
                           ('roc_auc', '#E74C3C', axes[1]),
                           ('brier', '#9B59B6', axes[2])]:
    ax.bar([f't+{h}h' for h in hm_df['horizon_h']], hm_df[metric], color=color, alpha=0.8)
    ax.set_ylim(0, 1); ax.set_ylabel(metric.upper()); ax.set_title(metric.upper())
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('eda_plots/16_forecaster_horizons.png', dpi=150, bbox_inches='tight')
plt.show()

## Log Validation Results to MLflow

In [ ]:
labeled_ds = mlflow.data.from_pandas(df, name='plant_alarm_labeled',
                                      source='data_drop/labeled_dataset.csv')

with mlflow.start_run(run_name='validation_suite') as run:
    mlflow.log_input(labeled_ds, context='validation')

    # Isolation Forest metrics
    mlflow.log_metrics({
        'if_roc_auc':        roc_auc_if,
        'if_avg_precision':  ap_if,
        'if_best_threshold': best_thr,
        'if_best_f1':        best_f1,
        'cv_mean_f1':        cv_df['f1'].mean(),
        'cv_std_f1':         cv_df['f1'].std(),
        'cv_mean_roc_auc':   cv_df['roc_auc'].mean(),
    })

    # Soft alarm blend metrics
    if lr_alarm_local and rf_alarm_local:
        from sklearn.metrics import roc_auc_score as roc_
        from sklearn.metrics import brier_score_loss
        blend_t = 0.5 * lr_alarm_local.predict_proba(X_test)[:, 1] + \
                  0.5 * rf_alarm_local.predict_proba(X_test)[:, 1]
        mlflow.log_metrics({
            'blend_roc_auc':    float(roc_(y_test, blend_t)),
            'blend_brier':      float(brier_score_loss(y_test, blend_t)),
        })

    # Forecaster metrics per horizon
    for _, row in hm_df.iterrows():
        h = int(row['horizon_h'])
        mlflow.log_metrics({
            f'fc{h}h_f1':       row['f1'],
            f'fc{h}h_roc_auc':  row['roc_auc'],
            f'fc{h}h_brier':    row['brier'],
            f'fc{h}h_precision':row['precision'],
            f'fc{h}h_recall':   row['recall'],
        })

    # Artefacts
    import tempfile, os as _os
    tmp = tempfile.gettempdir()
    cv_df.to_csv(_os.path.join(tmp, 'cv_results.csv'), index=False)
    hm_df.to_csv(_os.path.join(tmp, 'horizon_metrics.csv'), index=False)
    mlflow.log_artifact(_os.path.join(tmp, 'cv_results.csv'),      artifact_path='reports')
    mlflow.log_artifact(_os.path.join(tmp, 'horizon_metrics.csv'), artifact_path='reports')
    for plot in ['14_isolation_forest_curves', '15_threshold_sweep',
                 '16_forecaster_horizons', '17_soft_alarm_calibration']:
        path = f'eda_plots/{plot}.png'
        if _os.path.exists(path):
            mlflow.log_artifact(path, artifact_path='plots')

    VAL_RUN_ID = run.info.run_id
    print(f'Validation run logged: {VAL_RUN_ID}')

print('\nValidation complete.')